In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['dipBuy_db']
trades_collection  = db['trades_cardwell_rsi_v31']
exclude_collection = db['excluded_tickers_cardwell_rsi_v31']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS  — Cardwell RSI Range Shift
# ══════════════════════════════════════════════════════════════════════════════

RSI_PERIOD   = 14    # Standard RSI period

# ── Trend classification (daily RSI range over N bars) ──────────────────────
TREND_LOOKBACK = 50  # bars used to confirm RSI range regime

# ── Uptrend RSI range: RSI oscillates between 40–80 ─────────────────────────
UP_RSI_FLOOR   = 40  # RSI must stay above this to confirm uptrend
UP_RSI_CEIL    = 80  # RSI upper bound in uptrend

# ── Downtrend RSI range: RSI oscillates between 20–60 ───────────────────────
DN_RSI_FLOOR   = 20  # RSI lower bound in downtrend
DN_RSI_CEIL    = 60  # RSI must stay below this to confirm downtrend

# ── Entry zones ──────────────────────────────────────────────────────────────
LONG_ENTRY_LOW  = 40  # Buy when RSI pulls back into this zone …
LONG_ENTRY_HIGH = 50  # … uptrend support band

SHORT_ENTRY_LOW  = 50  # Sell when RSI rallies into this zone …
SHORT_ENTRY_HIGH = 60  # … downtrend resistance band

# ── Exit levels ──────────────────────────────────────────────────────────────
LONG_EXIT_RSI  = 70  # Close long when RSI reaches upper range
SHORT_EXIT_RSI = 30  # Cover short when RSI falls to lower range

# ── Misc ─────────────────────────────────────────────────────────────────────
ORDER_SIZE = 40      # Fixed shares per trade
MA_PERIOD  = 200     # Secondary trend confirmation (daily SMA)


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER  — always DAY to prevent TWS preset overriding TIF to GTC
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with tif='DAY' explicitly set.
    Prevents TWS order presets from silently converting TIF to GTC
    (error 10349), which causes the order to be cancelled immediately.
    """
    order     = MarketOrder(action, qty)
    order.tif = 'DAY'
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series: pd.Series, period: int = RSI_PERIOD) -> pd.Series:
    """Wilder-smoothed RSI."""
    d    = series.diff()
    gain = d.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
    loss = (-d.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
    return 100 - 100 / (1 + gain / (loss + 1e-10))


def classify_trend(rsi_series: pd.Series, lookback: int = TREND_LOOKBACK) -> str:
    """
    Cardwell Range Shift regime detection.

    Examine the last `lookback` RSI values:
      • If the RSI min ≥ UP_RSI_FLOOR (40) and max ≤ UP_RSI_CEIL (80)
        → UPTREND  (RSI locked in the 40–80 bullish range)
      • If the RSI min ≥ DN_RSI_FLOOR (20) and max ≤ DN_RSI_CEIL (60)
        → DOWNTREND  (RSI locked in the 20–60 bearish range)
      • Otherwise → NEUTRAL (transitioning / no clear range)

    A relaxed version scores each condition proportionally and picks
    whichever regime has more bars satisfying its bounds.
    """
    window = rsi_series.iloc[-lookback:]
    if len(window) < lookback:
        return 'NEUTRAL'

    up_bars = ((window >= UP_RSI_FLOOR) & (window <= UP_RSI_CEIL)).sum()
    dn_bars = ((window >= DN_RSI_FLOOR) & (window <= DN_RSI_CEIL)).sum()

    threshold = lookback * 0.70   # 70 % of bars must fit the range

    if up_bars >= threshold and dn_bars < threshold:
        return 'UPTREND'
    if dn_bars >= threshold and up_bars < threshold:
        return 'DOWNTREND'
    if up_bars >= threshold and dn_bars >= threshold:
        # Overlapping — use most recent RSI to break tie
        last = float(rsi_series.iloc[-1])
        return 'UPTREND' if last > 50 else 'DOWNTREND'
    return 'NEUTRAL'


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_data(contract, ma_period: int = MA_PERIOD, rsi_period: int = RSI_PERIOD,
                   trend_lookback: int = TREND_LOOKBACK):
    """
    Fetch daily bars sufficient to compute:
      • 200-day SMA  (secondary trend confirmation)
      • RSI(14) range regime  (Cardwell range shift)

    Returns (sma200, daily_close, regime) or (None, None, None) on failure.
    """
    needed = max(ma_period, rsi_period + trend_lookback) + 20
    bars   = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{needed} D',
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + trend_lookback:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI']  = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    sma200      = float(df['close'].rolling(ma_period).mean().iloc[-1]) if len(df) >= ma_period else None
    daily_close = float(df['close'].iloc[-1])
    regime      = classify_trend(df['RSI'], lookback=trend_lookback)

    return sma200, daily_close, regime


def get_intraday_rsi(contract, rsi_period: int = RSI_PERIOD):
    """
    Fetch 5-min bars and compute RSI(14) for entry/exit signals.
    Returns (price_now, rsi_now, rsi_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='5 D',           # enough warm-up bars for RSI(14)
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + 5:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI']  = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    price    = float(df.iloc[-1]['close'])
    rsi_now  = float(df.iloc[-1]['RSI'])
    rsi_prev = float(df.iloc[-2]['RSI'])
    return price, rsi_now, rsi_prev


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi_at_entry, regime, sma200):
    ep = float(entry_price)
    return {
        'instrument':      symbol,
        'direction':       direction,
        'entry_price':     ep,
        'quantity':        int(qty),
        'remaining_qty':   int(qty),
        'rsi_at_entry':    float(rsi_at_entry),
        'regime_at_entry': regime,
        'sma200_at_entry': float(sma200) if sma200 else None,
        'entry_timestamp': datetime.now(),
        'order_id':        str(ObjectId()),
        # ── Live mark-to-market ──────────────────────────────────────────
        # Long:  peak_price   = highest close seen while trade is open
        # Short: trough_price = lowest  close seen while trade is open
        'current_price':       ep,
        'current_pnl':         0.0,
        'current_pnl_pct':     0.0,
        'peak_price':          ep,    # most favourable price (long=high, short=low)
        'trough_price':        ep,    # least favourable price (long=low, short=high)
        'max_unrealized_pnl':  0.0,   # best open profit ever seen ($)
        'min_unrealized_pnl':  0.0,   # deepest open loss ever seen ($, negative)
        'last_mark_timestamp': datetime.now(),
        # ────────────────────────────────────────────────────────────────
        'exit_price':      None,
        'exit_timestamp':  None,
        'exit_rsi':        None,
        'reason':          None,
        'realized_pnl':    None,
        'status':          'open',
        'created_at':      datetime.now(),
        'updated_at':      datetime.now(),
    }


def db_update(trade_id, updates: dict):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})


def update_trade_marks(trade_doc: dict, current_price: float) -> dict:
    """
    Recompute and persist all live mark-to-market fields for an open trade.

    Tracked fields
    ──────────────
    current_price       — latest bar close
    current_pnl         — open P&L in dollars
    current_pnl_pct     — open P&L as a fraction of entry cost
    peak_price          — most favourable price seen during the trade
                          (highest close for longs, lowest close for shorts)
    trough_price        — least favourable price seen during the trade
                          (lowest close for longs, highest close for shorts)
    max_unrealized_pnl  — largest open profit ever observed ($)
    min_unrealized_pnl  — largest open loss ever observed ($, stored negative)
    last_mark_timestamp — time of this update
    """
    entry_price = float(trade_doc['entry_price'])
    qty         = int(trade_doc['remaining_qty'])
    direction   = trade_doc.get('direction', 'long')
    cp          = float(current_price)

    # ── Direction-aware P&L ───────────────────────────────────────────────────
    if direction == 'long':
        pnl = (cp - entry_price) * qty
    else:  # short
        pnl = (entry_price - cp) * qty

    pnl_pct = pnl / (entry_price * qty) if entry_price * qty != 0 else 0.0

    # ── Peak / trough price ───────────────────────────────────────────────────
    prev_peak   = float(trade_doc.get('peak_price',   entry_price))
    prev_trough = float(trade_doc.get('trough_price', entry_price))

    if direction == 'long':
        new_peak   = max(prev_peak,   cp)   # long peak   = highest price  (best profit)
        new_trough = min(prev_trough, cp)   # long trough = lowest  price  (deepest loss)
    else:
        new_peak   = min(prev_peak,   cp)   # short peak  = lowest  price  (best profit)
        new_trough = max(prev_trough, cp)   # short trough= highest price  (deepest loss)

    # ── Max / min unrealized P&L ($) ─────────────────────────────────────────
    prev_max_pnl = float(trade_doc.get('max_unrealized_pnl', 0.0))
    prev_min_pnl = float(trade_doc.get('min_unrealized_pnl', 0.0))
    new_max_pnl  = max(prev_max_pnl, pnl)
    new_min_pnl  = min(prev_min_pnl, pnl)

    marks = {
        'current_price':       round(cp,       4),
        'current_pnl':         round(pnl,      2),
        'current_pnl_pct':     round(pnl_pct,  6),
        'peak_price':          round(new_peak,  4),
        'trough_price':        round(new_trough,4),
        'max_unrealized_pnl':  round(new_max_pnl, 2),
        'min_unrealized_pnl':  round(new_min_pnl, 2),
        'last_mark_timestamp': datetime.now(),
    }
    db_update(trade_doc['_id'], marks)
    return marks


def do_sell(contract, qty, entry_price, sell_price, rsi, reason, trade_id):
    """Close long position."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl  = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi':       float(rsi),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
          f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


def do_cover(contract, qty, entry_price, cover_price, rsi, reason, trade_id):
    """Cover short position."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl  = (entry_price - cover_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi':       float(rsi),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
          f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  ANDREW CARDWELL — RSI RANGE SHIFT STRATEGY                             ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  REGIME DETECTION (daily RSI over last 50 bars)                         ║
    ║    UPTREND   — RSI oscillates within  40–80  (bullish range)            ║
    ║    DOWNTREND — RSI oscillates within  20–60  (bearish range)            ║
    ║    NEUTRAL   — no clear range, sit out                                  ║
    ║                                                                          ║
    ║  ENTRY  (5-min RSI for precision timing)                                 ║
    ║    Long  — regime=UPTREND   + RSI pulls back into 40–50 zone            ║
    ║    Short — regime=DOWNTREND + RSI rallies into  50–60 zone              ║
    ║                                                                          ║
    ║  EXIT                                                                    ║
    ║    Long  — RSI reaches 70  (upper Cardwell uptrend bound)               ║
    ║    Short — RSI falls to 30 (lower Cardwell downtrend bound)             ║
    ║                                                                          ║
    ║  Position size: 40 shares fixed                                          ║
    ║  One trade per ticker per day                                            ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── 1. Daily regime detection ─────────────────────────────────
            sma200, daily_close, regime = get_daily_data(contract)
            if regime is None:
                print(f"  Insufficient daily data — skipping {symbol}")
                continue

            sma_str = f"SMA200=${sma200:.2f}  " if sma200 else ""
            print(f"  {sma_str}DailyClose=${daily_close:.2f}  Regime={regime}")

            if regime == 'NEUTRAL':
                print(f"  RSI range is NEUTRAL — no clear trend, skipping.")
                continue

            # ── 2. Intraday RSI(14) for entry timing ──────────────────────
            price, rsi_now, rsi_prev = get_intraday_rsi(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                continue

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  RSI_prev={rsi_prev:.1f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty         = int(open_trade['remaining_qty'])
                direction   = open_trade.get('direction', 'long')
                trade_id    = open_trade['_id']

                if direction == 'long':
                    # ── Update live marks every scan ────────────────────
                    marks = update_trade_marks(open_trade, price)

                    print(f"  OPEN LONG  entry=${entry_price:.2f}  qty={qty}")
                    print(f"    Current : ${marks['current_price']:.2f}  "
                          f"P&L={marks['current_pnl_pct']:+.2%}  (${marks['current_pnl']:+.2f})")
                    print(f"    Peak    : ${marks['peak_price']:.2f}  "
                          f"MaxProfit=${marks['max_unrealized_pnl']:+.2f}")
                    print(f"    Trough  : ${marks['trough_price']:.2f}  "
                          f"MaxLoss  =${marks['min_unrealized_pnl']:+.2f}")

                    # Exit long: RSI reaches the upper Cardwell uptrend boundary
                    if rsi_now >= LONG_EXIT_RSI:
                        do_sell(contract, qty, entry_price, price,
                                rsi_now, f'RSI_REACHED_{LONG_EXIT_RSI}_LONG_EXIT', trade_id)
                        print(f"  📈 LONG EXIT: RSI={rsi_now:.1f} ≥ {LONG_EXIT_RSI}")

                    # Safety: regime flipped to downtrend — exit stale long
                    elif regime == 'DOWNTREND':
                        do_sell(contract, qty, entry_price, price,
                                rsi_now, 'REGIME_FLIP_TO_DOWNTREND', trade_id)
                        print(f"  ⚠️  LONG EXIT: regime flipped to DOWNTREND")

                    else:
                        print(f"  Holding long — waiting RSI ≥ {LONG_EXIT_RSI}  (RSI={rsi_now:.1f})")

                elif direction == 'short':
                    # ── Update live marks every scan ────────────────────
                    marks = update_trade_marks(open_trade, price)

                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}")
                    print(f"    Current : ${marks['current_price']:.2f}  "
                          f"P&L={marks['current_pnl_pct']:+.2%}  (${marks['current_pnl']:+.2f})")
                    print(f"    Peak    : ${marks['peak_price']:.2f}  "
                          f"MaxProfit=${marks['max_unrealized_pnl']:+.2f}")
                    print(f"    Trough  : ${marks['trough_price']:.2f}  "
                          f"MaxLoss  =${marks['min_unrealized_pnl']:+.2f}")

                    # Exit short: RSI falls to the lower Cardwell downtrend boundary
                    if rsi_now <= SHORT_EXIT_RSI:
                        do_cover(contract, qty, entry_price, price,
                                 rsi_now, f'RSI_FELL_TO_{SHORT_EXIT_RSI}_SHORT_EXIT', trade_id)
                        print(f"  📉 SHORT EXIT: RSI={rsi_now:.1f} ≤ {SHORT_EXIT_RSI}")

                    # Safety: regime flipped to uptrend — cover stale short
                    elif regime == 'UPTREND':
                        do_cover(contract, qty, entry_price, price,
                                 rsi_now, 'REGIME_FLIP_TO_UPTREND', trade_id)
                        print(f"  ⚠️  SHORT EXIT: regime flipped to UPTREND")

                    else:
                        print(f"  Holding short — waiting RSI ≤ {SHORT_EXIT_RSI}  (RSI={rsi_now:.1f})")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                in_long_zone  = LONG_ENTRY_LOW  <= rsi_now <= LONG_ENTRY_HIGH
                in_short_zone = SHORT_ENTRY_LOW <= rsi_now <= SHORT_ENTRY_HIGH

                # ── LONG: uptrend + RSI pulls back to 40–50 support ───────
                if regime == 'UPTREND' and in_long_zone:
                    ib.placeOrder(contract, day_market_order('BUY', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'long', price, ORDER_SIZE,
                                           rsi_now, regime, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 BUY (long)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  Regime=UPTREND")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI:    {rsi_now:.1f}  (pullback to 40–50 support zone)")
                    print(f"     Target: RSI ≥ {LONG_EXIT_RSI}")

                # ── SHORT: downtrend + RSI rallies to 50–60 resistance ────
                elif regime == 'DOWNTREND' and in_short_zone:
                    ib.placeOrder(contract, day_market_order('SELL', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'short', price, ORDER_SIZE,
                                           rsi_now, regime, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🔻 SELL (short)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  Regime=DOWNTREND")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI:    {rsi_now:.1f}  (rally to 50–60 resistance zone)")
                    print(f"     Target: RSI ≤ {SHORT_EXIT_RSI}")

                else:
                    if regime == 'UPTREND':
                        print(f"  No entry — UPTREND but RSI={rsi_now:.1f}"
                              f"  (waiting for 40–50 pullback zone)")
                    else:
                        print(f"  No entry — DOWNTREND but RSI={rsi_now:.1f}"
                              f"  (waiting for 50–60 rally zone)")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['dipBuy_db']
trades_collection  = db['trades_cardwell_rsi_v2']
exclude_collection = db['excluded_tickers_cardwell_rsi_v2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS — Cardwell RSI Range Shift
# ══════════════════════════════════════════════════════════════════════════════

RSI_PERIOD = 14

# ── Trend classification (daily RSI range over N bars) ──────────────────────
TREND_LOOKBACK = 50

# ── Uptrend RSI range: RSI oscillates between 40–80 ─────────────────────────
UP_RSI_FLOOR = 40
UP_RSI_CEIL  = 80

# ── Downtrend RSI range: RSI oscillates between 20–60 ───────────────────────
DN_RSI_FLOOR = 20
DN_RSI_CEIL  = 60

# ── Entry zones ──────────────────────────────────────────────────────────────
LONG_ENTRY_LOW   = 40
LONG_ENTRY_HIGH  = 50

SHORT_ENTRY_LOW  = 50
SHORT_ENTRY_HIGH = 60

# ── Exit levels ──────────────────────────────────────────────────────────────
LONG_EXIT_RSI  = 70
SHORT_EXIT_RSI = 30

# ── Risk management ──────────────────────────────────────────────────────────
HARD_STOP_LOSS            = -100.0   # exit if open PnL <= -100
BREAKEVEN_ARM_PNL         = 100.0    # once max PnL reaches this...
BREAKEVEN_EXIT_PNL        = 0.0      # ...do not allow trade to go red
LOCK_PROFIT_ARM_PNL       = 150.0    # once max PnL reaches this...
LOCK_PROFIT_EXIT_PNL      = 50.0     # ...keep at least +$50
TRAILING_ARM_PNL          = 150.0    # start trailing after this peak profit
TRAILING_GIVEBACK_DOLLARS = 100.0    # exit after giving back $100 from peak

# ── Position sizing ──────────────────────────────────────────────────────────
TARGET_POSITION_VALUE = 4000.0   # use roughly $4k notional per trade
MIN_SHARES            = 1
MAX_SHARES            = 250

# ── Misc ─────────────────────────────────────────────────────────────────────
MA_PERIOD = 200


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER — always DAY to prevent TWS preset overriding TIF to GTC
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with tif='DAY' explicitly set.
    Prevents TWS order presets from silently converting TIF to GTC
    (error 10349), which causes the order to be cancelled immediately.
    """
    order = MarketOrder(action, qty)
    order.tif = 'DAY'
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS / SANITIZERS
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy-like types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


def to_object_id(value):
    return value if isinstance(value, ObjectId) else ObjectId(value)


def db_update(trade_id, updates: dict):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': to_object_id(trade_id)}, {'$set': updates})


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series: pd.Series, period: int = RSI_PERIOD) -> pd.Series:
    """Wilder-style RSI."""
    delta = series.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = gain / (loss + 1e-10)
    return 100 - (100 / (1 + rs))


def classify_trend(rsi_series: pd.Series, lookback: int = TREND_LOOKBACK) -> str:
    """
    Cardwell Range Shift regime detection.

    Examine the last `lookback` RSI values:
      • If RSI mostly fits 40–80 → UPTREND
      • If RSI mostly fits 20–60 → DOWNTREND
      • Otherwise → NEUTRAL
    """
    window = rsi_series.iloc[-lookback:]
    if len(window) < lookback:
        return 'NEUTRAL'

    up_bars = ((window >= UP_RSI_FLOOR) & (window <= UP_RSI_CEIL)).sum()
    dn_bars = ((window >= DN_RSI_FLOOR) & (window <= DN_RSI_CEIL)).sum()

    threshold = lookback * 0.70

    if up_bars >= threshold and dn_bars < threshold:
        return 'UPTREND'
    if dn_bars >= threshold and up_bars < threshold:
        return 'DOWNTREND'
    if up_bars >= threshold and dn_bars >= threshold:
        last = float(rsi_series.iloc[-1])
        return 'UPTREND' if last > 50 else 'DOWNTREND'
    return 'NEUTRAL'


# ══════════════════════════════════════════════════════════════════════════════
# POSITION SIZING
# ══════════════════════════════════════════════════════════════════════════════

def calculate_order_size(price: float) -> int:
    """
    Dollar-based position sizing to avoid huge exposure differences
    across cheap vs expensive tickers.
    """
    if price is None or price <= 0:
        return 0

    qty = int(TARGET_POSITION_VALUE // price)
    qty = max(MIN_SHARES, qty)
    qty = min(MAX_SHARES, qty)
    return qty


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_data(contract, ma_period: int = MA_PERIOD, rsi_period: int = RSI_PERIOD,
                   trend_lookback: int = TREND_LOOKBACK):
    """
    Fetch daily bars sufficient to compute:
      • 200-day SMA
      • RSI(14) range regime

    Returns (sma200, daily_close, regime) or (None, None, None) on failure.
    """
    needed = max(ma_period, rsi_period + trend_lookback) + 20
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{needed} D',
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + trend_lookback:
        return None, None, None

    df = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI'] = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    sma200 = float(df['close'].rolling(ma_period).mean().iloc[-1]) if len(df) >= ma_period else None
    daily_close = float(df['close'].iloc[-1])
    regime = classify_trend(df['RSI'], lookback=trend_lookback)

    return sma200, daily_close, regime


def get_intraday_rsi(contract, rsi_period: int = RSI_PERIOD):
    """
    Fetch 5-min bars and compute RSI(14) for entry/exit signals.
    Returns (price_now, rsi_now, rsi_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='5 D',
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + 5:
        return None, None, None

    df = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI'] = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    price = float(df.iloc[-1]['close'])
    rsi_now = float(df.iloc[-1]['RSI'])
    rsi_prev = float(df.iloc[-2]['RSI'])
    return price, rsi_now, rsi_prev


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENTS / MARKS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi_at_entry, regime, sma200):
    ep = float(entry_price)
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': ep,
        'quantity': int(qty),
        'remaining_qty': int(qty),
        'position_value_at_entry': round(ep * qty, 2),
        'rsi_at_entry': float(rsi_at_entry),
        'regime_at_entry': regime,
        'sma200_at_entry': float(sma200) if sma200 is not None else None,
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),

        # Live mark-to-market
        'current_price': ep,
        'current_pnl': 0.0,
        'current_pnl_pct': 0.0,
        'peak_price': ep,
        'trough_price': ep,
        'max_unrealized_pnl': 0.0,
        'min_unrealized_pnl': 0.0,
        'last_mark_timestamp': datetime.now(),

        # Risk model used
        'risk_rules': {
            'hard_stop_loss': HARD_STOP_LOSS,
            'breakeven_arm_pnl': BREAKEVEN_ARM_PNL,
            'breakeven_exit_pnl': BREAKEVEN_EXIT_PNL,
            'lock_profit_arm_pnl': LOCK_PROFIT_ARM_PNL,
            'lock_profit_exit_pnl': LOCK_PROFIT_EXIT_PNL,
            'trailing_arm_pnl': TRAILING_ARM_PNL,
            'trailing_giveback_dollars': TRAILING_GIVEBACK_DOLLARS,
            'target_position_value': TARGET_POSITION_VALUE,
        },

        # Exit fields
        'exit_price': None,
        'exit_timestamp': None,
        'exit_rsi': None,
        'reason': None,
        'realized_pnl': None,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now(),
    }


def update_trade_marks(trade_doc: dict, current_price: float) -> dict:
    """
    Recompute and persist all live mark-to-market fields for an open trade.
    """
    entry_price = float(trade_doc['entry_price'])
    qty = int(trade_doc['remaining_qty'])
    direction = trade_doc.get('direction', 'long')
    cp = float(current_price)

    if direction == 'long':
        pnl = (cp - entry_price) * qty
    else:
        pnl = (entry_price - cp) * qty

    pnl_pct = pnl / (entry_price * qty) if entry_price * qty != 0 else 0.0

    prev_peak = float(trade_doc.get('peak_price', entry_price))
    prev_trough = float(trade_doc.get('trough_price', entry_price))

    if direction == 'long':
        new_peak = max(prev_peak, cp)
        new_trough = min(prev_trough, cp)
    else:
        new_peak = min(prev_peak, cp)
        new_trough = max(prev_trough, cp)

    prev_max_pnl = float(trade_doc.get('max_unrealized_pnl', 0.0))
    prev_min_pnl = float(trade_doc.get('min_unrealized_pnl', 0.0))
    new_max_pnl = max(prev_max_pnl, pnl)
    new_min_pnl = min(prev_min_pnl, pnl)

    marks = {
        'current_price': round(cp, 4),
        'current_pnl': round(pnl, 2),
        'current_pnl_pct': round(pnl_pct, 6),
        'peak_price': round(new_peak, 4),
        'trough_price': round(new_trough, 4),
        'max_unrealized_pnl': round(new_max_pnl, 2),
        'min_unrealized_pnl': round(new_min_pnl, 2),
        'last_mark_timestamp': datetime.now(),
    }
    db_update(trade_doc['_id'], marks)
    return marks


# ══════════════════════════════════════════════════════════════════════════════
# EXIT MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_exit(direction: str, marks: dict, rsi_now: float, regime: str):
    """
    3-layer exit management + original RSI / regime exits.

    Priority:
      1) Hard stop-loss
      2) Breakeven protection
      3) Lock-profit protection
      4) Trailing giveback stop
      5) RSI target exit
      6) Daily regime flip exit
    """
    current_pnl = float(marks['current_pnl'])
    max_pnl = float(marks['max_unrealized_pnl'])

    # Layer 1 — hard stop-loss
    if current_pnl <= HARD_STOP_LOSS:
        return (
            f'HARD_STOP_LOSS current_pnl={current_pnl:.2f} threshold={HARD_STOP_LOSS:.2f}',
            f'Hard stop hit: current_pnl ${current_pnl:.2f} <= ${HARD_STOP_LOSS:.2f}'
        )

    # Layer 2A — breakeven protection
    if max_pnl >= BREAKEVEN_ARM_PNL and current_pnl <= BREAKEVEN_EXIT_PNL:
        return (
            f'BREAKEVEN_PROTECTION max_pnl={max_pnl:.2f} current_pnl={current_pnl:.2f}',
            f'Breakeven protection: max_pnl ${max_pnl:.2f} and current_pnl ${current_pnl:.2f} <= ${BREAKEVEN_EXIT_PNL:.2f}'
        )

    # Layer 2B — lock-profit logic
    if max_pnl >= LOCK_PROFIT_ARM_PNL and current_pnl <= LOCK_PROFIT_EXIT_PNL:
        return (
            f'LOCK_PROFIT_PROTECTION max_pnl={max_pnl:.2f} current_pnl={current_pnl:.2f}',
            f'Lock-profit exit: peak ${max_pnl:.2f}, current ${current_pnl:.2f} <= ${LOCK_PROFIT_EXIT_PNL:.2f}'
        )

    # Layer 3 — trailing giveback stop
    if max_pnl >= TRAILING_ARM_PNL:
        trailing_floor = max_pnl - TRAILING_GIVEBACK_DOLLARS
        if current_pnl <= trailing_floor:
            return (
                f'TRAILING_GIVEBACK_STOP max_pnl={max_pnl:.2f} current_pnl={current_pnl:.2f} trailing_floor={trailing_floor:.2f}',
                f'Trailing giveback stop: peak ${max_pnl:.2f}, current ${current_pnl:.2f}, floor ${trailing_floor:.2f}'
            )

    # Original RSI exits
    if direction == 'long' and rsi_now >= LONG_EXIT_RSI:
        return (
            f'RSI_REACHED_{LONG_EXIT_RSI}_LONG_EXIT',
            f'Long RSI exit: RSI {rsi_now:.1f} >= {LONG_EXIT_RSI}'
        )

    if direction == 'short' and rsi_now <= SHORT_EXIT_RSI:
        return (
            f'RSI_FELL_TO_{SHORT_EXIT_RSI}_SHORT_EXIT',
            f'Short RSI exit: RSI {rsi_now:.1f} <= {SHORT_EXIT_RSI}'
        )

    # Daily regime flip exits
    if direction == 'long' and regime == 'DOWNTREND':
        return (
            'REGIME_FLIP_TO_DOWNTREND',
            'Long exit: daily regime flipped to DOWNTREND'
        )

    if direction == 'short' and regime == 'UPTREND':
        return (
            'REGIME_FLIP_TO_UPTREND',
            'Short exit: daily regime flipped to UPTREND'
        )

    return None, None


def do_sell(contract, qty, entry_price, sell_price, rsi, reason, trade_id):
    """Close long position."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price': float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi': float(rsi),
        'reason': reason,
        'realized_pnl': float(pnl),
        'status': 'closed',
        'remaining_qty': 0,
    })
    print(
        f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
        f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}"
    )


def do_cover(contract, qty, entry_price, cover_price, rsi, reason, trade_id):
    """Cover short position."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl = (entry_price - cover_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price': float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi': float(rsi),
        'reason': reason,
        'realized_pnl': float(pnl),
        'status': 'closed',
        'remaining_qty': 0,
    })
    print(
        f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
        f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    Andrew Cardwell RSI Range Shift strategy with:
      • dollar-based position sizing
      • hard stop-loss
      • breakeven / lock-profit protection
      • trailing giveback stop
      • RSI exits
      • regime-flip exits
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ── 1. Daily regime detection ─────────────────────────────────
            sma200, daily_close, regime = get_daily_data(contract)
            if regime is None:
                print("  Daily data unavailable / insufficient.")
            else:
                sma_str = f"SMA200=${sma200:.2f}  " if sma200 is not None else ""
                print(f"  {sma_str}DailyClose=${daily_close:.2f}  Regime={regime}")

            # ── 2. Intraday RSI(14) for entry timing / risk management ───
            price, rsi_now, rsi_prev = get_intraday_rsi(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                await asyncio.sleep(0.5)
                continue

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  RSI_prev={rsi_prev:.1f}")

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════
            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty = int(open_trade['remaining_qty'])
                direction = open_trade.get('direction', 'long')
                trade_id = open_trade['_id']

                marks = update_trade_marks(open_trade, price)

                if direction == 'long':
                    print(f"  OPEN LONG   entry=${entry_price:.2f}  qty={qty}")
                else:
                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}")

                print(
                    f"    Current : ${marks['current_price']:.2f}  "
                    f"P&L={marks['current_pnl_pct']:+.2%}  (${marks['current_pnl']:+.2f})"
                )
                print(
                    f"    Peak    : ${marks['peak_price']:.2f}  "
                    f"MaxProfit=${marks['max_unrealized_pnl']:+.2f}"
                )
                print(
                    f"    Trough  : ${marks['trough_price']:.2f}  "
                    f"MaxLoss  =${marks['min_unrealized_pnl']:+.2f}"
                )

                exit_reason, exit_msg = evaluate_exit(direction, marks, rsi_now, regime)

                if exit_reason:
                    if direction == 'long':
                        do_sell(contract, qty, entry_price, price, rsi_now, exit_reason, trade_id)
                    else:
                        do_cover(contract, qty, entry_price, price, rsi_now, exit_reason, trade_id)

                    print(f"  🚨 EXIT TRIGGER: {exit_msg}")
                else:
                    if direction == 'long':
                        print(
                            f"  Holding long — no exit condition hit "
                            f"(hard stop {HARD_STOP_LOSS}, RSI target {LONG_EXIT_RSI})"
                        )
                    else:
                        print(
                            f"  Holding short — no exit condition hit "
                            f"(hard stop {HARD_STOP_LOSS}, RSI target {SHORT_EXIT_RSI})"
                        )

                await asyncio.sleep(0.5)
                continue

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            # No new trade if daily regime is unavailable / neutral
            if regime is None:
                print("  No entry — daily regime unavailable.")
                await asyncio.sleep(0.5)
                continue

            if regime == 'NEUTRAL':
                print("  No entry — RSI daily range is NEUTRAL.")
                await asyncio.sleep(0.5)
                continue

            today = datetime.now().date()
            if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                print(f"  Already traded {symbol} today — skipping.")
                await asyncio.sleep(0.5)
                continue

            in_long_zone = LONG_ENTRY_LOW <= rsi_now <= LONG_ENTRY_HIGH
            in_short_zone = SHORT_ENTRY_LOW <= rsi_now <= SHORT_ENTRY_HIGH

            qty = calculate_order_size(price)
            if qty <= 0:
                print(f"  Invalid order size for {symbol} at price ${price:.2f} — skipping.")
                await asyncio.sleep(0.5)
                continue

            position_value = qty * price

            # ── LONG: uptrend + RSI pulls back to 40–50 support ──────────
            if regime == 'UPTREND' and in_long_zone:
                ib.placeOrder(contract, day_market_order('BUY', qty))

                doc = create_trade_doc(symbol, 'long', price, qty, rsi_now, regime, sma200)
                trades_collection.insert_one(sanitize_for_mongo(doc))
                exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                print(f"  🚀 BUY (long)  {symbol}")
                print(f"     Entry:    ${price:.2f}  |  Regime=UPTREND")
                print(f"     Shares:   {qty}")
                print(f"     Notional: ${position_value:.2f}")
                print(f"     RSI:      {rsi_now:.1f}  (pullback to 40–50 support zone)")
                print(f"     Risk:     Hard stop at ${HARD_STOP_LOSS:.2f}")
                print(f"     Target:   RSI ≥ {LONG_EXIT_RSI}")

            # ── SHORT: downtrend + RSI rallies to 50–60 resistance ───────
            elif regime == 'DOWNTREND' and in_short_zone:
                ib.placeOrder(contract, day_market_order('SELL', qty))

                doc = create_trade_doc(symbol, 'short', price, qty, rsi_now, regime, sma200)
                trades_collection.insert_one(sanitize_for_mongo(doc))
                exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                print(f"  🔻 SELL (short)  {symbol}")
                print(f"     Entry:    ${price:.2f}  |  Regime=DOWNTREND")
                print(f"     Shares:   {qty}")
                print(f"     Notional: ${position_value:.2f}")
                print(f"     RSI:      {rsi_now:.1f}  (rally to 50–60 resistance zone)")
                print(f"     Risk:     Hard stop at ${HARD_STOP_LOSS:.2f}")
                print(f"     Target:   RSI ≤ {SHORT_EXIT_RSI}")

            else:
                if regime == 'UPTREND':
                    print(
                        f"  No entry — UPTREND but RSI={rsi_now:.1f} "
                        f"(waiting for 40–50 pullback zone)"
                    )
                else:
                    print(
                        f"  No entry — DOWNTREND but RSI={rsi_now:.1f} "
                        f"(waiting for 50–60 rally zone)"
                    )

            await asyncio.sleep(0.5)

        print(f"\n{'='*60}")
        print("  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")

In [2]:
import asyncio
from ib_insync import *
import pandas as pd
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['dipBuy_db']
trades_collection  = db['trades_cardwell_rsi_v2']
exclude_collection = db['excluded_tickers_cardwell_rsi_v2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS — Cardwell RSI Range Shift
# ══════════════════════════════════════════════════════════════════════════════

RSI_PERIOD = 14

# ── Trend classification (daily RSI range over N bars) ──────────────────────
TREND_LOOKBACK = 50

# ── Uptrend RSI range: RSI oscillates between 40–80 ─────────────────────────
UP_RSI_FLOOR = 40
UP_RSI_CEIL  = 80

# ── Downtrend RSI range: RSI oscillates between 20–60 ───────────────────────
DN_RSI_FLOOR = 20
DN_RSI_CEIL  = 60

# ── Entry zones ──────────────────────────────────────────────────────────────
LONG_ENTRY_LOW   = 40
LONG_ENTRY_HIGH  = 50

SHORT_ENTRY_LOW  = 50
SHORT_ENTRY_HIGH = 60

# ── Exit levels ──────────────────────────────────────────────────────────────
LONG_EXIT_RSI  = 70
SHORT_EXIT_RSI = 30

# ── Risk management ──────────────────────────────────────────────────────────
HARD_STOP_LOSS            = -100.0   # exit if open PnL <= -100
BREAKEVEN_ARM_PNL         = 100.0    # once max PnL reaches this...
BREAKEVEN_EXIT_PNL        = 0.0      # ...do not allow trade to go red
LOCK_PROFIT_ARM_PNL       = 150.0    # once max PnL reaches this...
LOCK_PROFIT_EXIT_PNL      = 50.0     # ...keep at least +$50
TRAILING_ARM_PNL          = 150.0    # start trailing after this peak profit
TRAILING_GIVEBACK_DOLLARS = 100.0    # exit after giving back $100 from peak

# ── Position sizing ──────────────────────────────────────────────────────────
TARGET_POSITION_VALUE = 4000.0   # use roughly $4k notional per trade
MIN_SHARES            = 1
MAX_SHARES            = 250

# ── Misc ─────────────────────────────────────────────────────────────────────
MA_PERIOD = 200


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER — DAY + outsideRth for pre-market and regular hours
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with:
      - tif='DAY'        → prevents TWS preset silently converting TIF to GTC
                           (error 10349), which causes immediate cancellation.
      - outsideRth=True  → allows fills during pre-market (4am–9:30am ET)
                           and post-market (4pm–8pm ET) in addition to
                           regular trading hours.

    NOTE: Market orders in pre/post-market carry wide-spread risk.
    Ensure your IB account has 'Outside RTH' permissions enabled under
    Account Settings → Trading Permissions.
    """
    order = MarketOrder(action, qty)
    order.tif = 'DAY'
    order.outsideRth = True
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS / SANITIZERS
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy-like types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


def to_object_id(value):
    return value if isinstance(value, ObjectId) else ObjectId(value)


def db_update(trade_id, updates: dict):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': to_object_id(trade_id)}, {'$set': updates})


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series: pd.Series, period: int = RSI_PERIOD) -> pd.Series:
    """Wilder-style RSI."""
    delta = series.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = gain / (loss + 1e-10)
    return 100 - (100 / (1 + rs))


def classify_trend(rsi_series: pd.Series, lookback: int = TREND_LOOKBACK) -> str:
    """
    Cardwell Range Shift regime detection.

    Examine the last `lookback` RSI values:
      • If RSI mostly fits 40–80 → UPTREND
      • If RSI mostly fits 20–60 → DOWNTREND
      • Otherwise → NEUTRAL
    """
    window = rsi_series.iloc[-lookback:]
    if len(window) < lookback:
        return 'NEUTRAL'

    up_bars = ((window >= UP_RSI_FLOOR) & (window <= UP_RSI_CEIL)).sum()
    dn_bars = ((window >= DN_RSI_FLOOR) & (window <= DN_RSI_CEIL)).sum()

    threshold = lookback * 0.70

    if up_bars >= threshold and dn_bars < threshold:
        return 'UPTREND'
    if dn_bars >= threshold and up_bars < threshold:
        return 'DOWNTREND'
    if up_bars >= threshold and dn_bars >= threshold:
        last = float(rsi_series.iloc[-1])
        return 'UPTREND' if last > 50 else 'DOWNTREND'
    return 'NEUTRAL'


# ══════════════════════════════════════════════════════════════════════════════
# POSITION SIZING
# ══════════════════════════════════════════════════════════════════════════════

def calculate_order_size(price: float) -> int:
    """
    Dollar-based position sizing to avoid huge exposure differences
    across cheap vs expensive tickers.
    """
    if price is None or price <= 0:
        return 0

    qty = int(TARGET_POSITION_VALUE // price)
    qty = max(MIN_SHARES, qty)
    qty = min(MAX_SHARES, qty)
    return qty


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_data(contract, ma_period: int = MA_PERIOD, rsi_period: int = RSI_PERIOD,
                   trend_lookback: int = TREND_LOOKBACK):
    """
    Fetch daily bars sufficient to compute:
      • 200-day SMA
      • RSI(14) range regime

    useRTH=True is intentional — regime/SMA should be based on
    regular-hours closes only, not extended-hours noise.

    Returns (sma200, daily_close, regime) or (None, None, None) on failure.
    """
    needed = max(ma_period, rsi_period + trend_lookback) + 20
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{needed} D',
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,       # intentionally True — regime based on RTH closes
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + trend_lookback:
        return None, None, None

    df = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI'] = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    sma200 = float(df['close'].rolling(ma_period).mean().iloc[-1]) if len(df) >= ma_period else None
    daily_close = float(df['close'].iloc[-1])
    regime = classify_trend(df['RSI'], lookback=trend_lookback)

    return sma200, daily_close, regime


def get_intraday_rsi(contract, rsi_period: int = RSI_PERIOD):
    """
    Fetch 5-min bars (including pre/post-market) and compute RSI(14)
    for entry/exit signals.

    useRTH=False is intentional — captures pre-market price action
    so signals fire before the regular session open.

    Returns (price_now, rsi_now, rsi_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='5 D',
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,      # intentionally False — include pre/post-market bars
        formatDate=1
    )
    if not bars or len(bars) < rsi_period + 5:
        return None, None, None

    df = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df['RSI'] = calc_rsi(df['close'], period=rsi_period)
    df.dropna(subset=['RSI'], inplace=True)

    price = float(df.iloc[-1]['close'])
    rsi_now = float(df.iloc[-1]['RSI'])
    rsi_prev = float(df.iloc[-2]['RSI'])
    return price, rsi_now, rsi_prev


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENTS / MARKS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi_at_entry, regime, sma200):
    ep = float(entry_price)
    return {
        'instrument': symbol,
        'direction': direction,
        'entry_price': ep,
        'quantity': int(qty),
        'remaining_qty': int(qty),
        'position_value_at_entry': round(ep * qty, 2),
        'rsi_at_entry': float(rsi_at_entry),
        'regime_at_entry': regime,
        'sma200_at_entry': float(sma200) if sma200 is not None else None,
        'entry_timestamp': datetime.now(),
        'order_id': str(ObjectId()),

        # Live mark-to-market
        'current_price': ep,
        'current_pnl': 0.0,
        'current_pnl_pct': 0.0,
        'peak_price': ep,
        'trough_price': ep,
        'max_unrealized_pnl': 0.0,
        'min_unrealized_pnl': 0.0,
        'last_mark_timestamp': datetime.now(),

        # Risk model used
        'risk_rules': {
            'hard_stop_loss': HARD_STOP_LOSS,
            'breakeven_arm_pnl': BREAKEVEN_ARM_PNL,
            'breakeven_exit_pnl': BREAKEVEN_EXIT_PNL,
            'lock_profit_arm_pnl': LOCK_PROFIT_ARM_PNL,
            'lock_profit_exit_pnl': LOCK_PROFIT_EXIT_PNL,
            'trailing_arm_pnl': TRAILING_ARM_PNL,
            'trailing_giveback_dollars': TRAILING_GIVEBACK_DOLLARS,
            'target_position_value': TARGET_POSITION_VALUE,
        },

        # Exit fields
        'exit_price': None,
        'exit_timestamp': None,
        'exit_rsi': None,
        'reason': None,
        'realized_pnl': None,
        'status': 'open',
        'created_at': datetime.now(),
        'updated_at': datetime.now(),
    }


def update_trade_marks(trade_doc: dict, current_price: float) -> dict:
    """
    Recompute and persist all live mark-to-market fields for an open trade.
    """
    entry_price = float(trade_doc['entry_price'])
    qty = int(trade_doc['remaining_qty'])
    direction = trade_doc.get('direction', 'long')
    cp = float(current_price)

    if direction == 'long':
        pnl = (cp - entry_price) * qty
    else:
        pnl = (entry_price - cp) * qty

    pnl_pct = pnl / (entry_price * qty) if entry_price * qty != 0 else 0.0

    prev_peak = float(trade_doc.get('peak_price', entry_price))
    prev_trough = float(trade_doc.get('trough_price', entry_price))

    if direction == 'long':
        new_peak = max(prev_peak, cp)
        new_trough = min(prev_trough, cp)
    else:
        new_peak = min(prev_peak, cp)
        new_trough = max(prev_trough, cp)

    prev_max_pnl = float(trade_doc.get('max_unrealized_pnl', 0.0))
    prev_min_pnl = float(trade_doc.get('min_unrealized_pnl', 0.0))
    new_max_pnl = max(prev_max_pnl, pnl)
    new_min_pnl = min(prev_min_pnl, pnl)

    marks = {
        'current_price': round(cp, 4),
        'current_pnl': round(pnl, 2),
        'current_pnl_pct': round(pnl_pct, 6),
        'peak_price': round(new_peak, 4),
        'trough_price': round(new_trough, 4),
        'max_unrealized_pnl': round(new_max_pnl, 2),
        'min_unrealized_pnl': round(new_min_pnl, 2),
        'last_mark_timestamp': datetime.now(),
    }
    db_update(trade_doc['_id'], marks)
    return marks


# ══════════════════════════════════════════════════════════════════════════════
# EXIT MANAGEMENT
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_exit(direction: str, marks: dict, rsi_now: float, regime: str):
    """
    3-layer exit management + original RSI / regime exits.

    Priority:
      1) Hard stop-loss
      2) Breakeven protection
      3) Lock-profit protection
      4) Trailing giveback stop
      5) RSI target exit
      6) Daily regime flip exit
    """
    current_pnl = float(marks['current_pnl'])
    max_pnl = float(marks['max_unrealized_pnl'])

    # Layer 1 — hard stop-loss
    if current_pnl <= HARD_STOP_LOSS:
        return (
            f'HARD_STOP_LOSS current_pnl={current_pnl:.2f} threshold={HARD_STOP_LOSS:.2f}',
            f'Hard stop hit: current_pnl ${current_pnl:.2f} <= ${HARD_STOP_LOSS:.2f}'
        )

    # Layer 2A — breakeven protection
    if max_pnl >= BREAKEVEN_ARM_PNL and current_pnl <= BREAKEVEN_EXIT_PNL:
        return (
            f'BREAKEVEN_PROTECTION max_pnl={max_pnl:.2f} current_pnl={current_pnl:.2f}',
            f'Breakeven protection: max_pnl ${max_pnl:.2f} and current_pnl ${current_pnl:.2f} <= ${BREAKEVEN_EXIT_PNL:.2f}'
        )

    # Layer 2B — lock-profit logic
    if max_pnl >= LOCK_PROFIT_ARM_PNL and current_pnl <= LOCK_PROFIT_EXIT_PNL:
        return (
            f'LOCK_PROFIT_PROTECTION max_pnl={max_pnl:.2f} current_pnl={current_pnl:.2f}',
            f'Lock-profit exit: peak ${max_pnl:.2f}, current ${current_pnl:.2f} <= ${LOCK_PROFIT_EXIT_PNL:.2f}'
        )

    # Layer 3 — trailing giveback stop
    if max_pnl >= TRAILING_ARM_PNL:
        trailing_floor = max_pnl - TRAILING_GIVEBACK_DOLLARS
        if current_pnl <= trailing_floor:
            return (
                f'TRAILING_GIVEBACK_STOP max_pnl={max_pnl:.2f} current_pnl={current_pnl:.2f} trailing_floor={trailing_floor:.2f}',
                f'Trailing giveback stop: peak ${max_pnl:.2f}, current ${current_pnl:.2f}, floor ${trailing_floor:.2f}'
            )

    # Original RSI exits
    if direction == 'long' and rsi_now >= LONG_EXIT_RSI:
        return (
            f'RSI_REACHED_{LONG_EXIT_RSI}_LONG_EXIT',
            f'Long RSI exit: RSI {rsi_now:.1f} >= {LONG_EXIT_RSI}'
        )

    if direction == 'short' and rsi_now <= SHORT_EXIT_RSI:
        return (
            f'RSI_FELL_TO_{SHORT_EXIT_RSI}_SHORT_EXIT',
            f'Short RSI exit: RSI {rsi_now:.1f} <= {SHORT_EXIT_RSI}'
        )

    # Daily regime flip exits
    if direction == 'long' and regime == 'DOWNTREND':
        return (
            'REGIME_FLIP_TO_DOWNTREND',
            'Long exit: daily regime flipped to DOWNTREND'
        )

    if direction == 'short' and regime == 'UPTREND':
        return (
            'REGIME_FLIP_TO_UPTREND',
            'Short exit: daily regime flipped to UPTREND'
        )

    return None, None


def do_sell(contract, qty, entry_price, sell_price, rsi, reason, trade_id):
    """Close long position."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price': float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi': float(rsi),
        'reason': reason,
        'realized_pnl': float(pnl),
        'status': 'closed',
        'remaining_qty': 0,
    })
    print(
        f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
        f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}"
    )


def do_cover(contract, qty, entry_price, cover_price, rsi, reason, trade_id):
    """Cover short position."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl = (entry_price - cover_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price': float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi': float(rsi),
        'reason': reason,
        'realized_pnl': float(pnl),
        'status': 'closed',
        'remaining_qty': 0,
    })
    print(
        f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
        f"  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}"
    )


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    Andrew Cardwell RSI Range Shift strategy with:
      • dollar-based position sizing
      • hard stop-loss
      • breakeven / lock-profit protection
      • trailing giveback stop
      • RSI exits
      • regime-flip exits
      • pre-market + regular hours order execution (outsideRth=True)
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ── 1. Daily regime detection ─────────────────────────────────
            sma200, daily_close, regime = get_daily_data(contract)
            if regime is None:
                print("  Daily data unavailable / insufficient.")
            else:
                sma_str = f"SMA200=${sma200:.2f}  " if sma200 is not None else ""
                print(f"  {sma_str}DailyClose=${daily_close:.2f}  Regime={regime}")

            # ── 2. Intraday RSI(14) for entry timing / risk management ───
            price, rsi_now, rsi_prev = get_intraday_rsi(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                await asyncio.sleep(0.5)
                continue

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  RSI_prev={rsi_prev:.1f}")

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════
            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty = int(open_trade['remaining_qty'])
                direction = open_trade.get('direction', 'long')
                trade_id = open_trade['_id']

                marks = update_trade_marks(open_trade, price)

                if direction == 'long':
                    print(f"  OPEN LONG   entry=${entry_price:.2f}  qty={qty}")
                else:
                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}")

                print(
                    f"    Current : ${marks['current_price']:.2f}  "
                    f"P&L={marks['current_pnl_pct']:+.2%}  (${marks['current_pnl']:+.2f})"
                )
                print(
                    f"    Peak    : ${marks['peak_price']:.2f}  "
                    f"MaxProfit=${marks['max_unrealized_pnl']:+.2f}"
                )
                print(
                    f"    Trough  : ${marks['trough_price']:.2f}  "
                    f"MaxLoss  =${marks['min_unrealized_pnl']:+.2f}"
                )

                exit_reason, exit_msg = evaluate_exit(direction, marks, rsi_now, regime)

                if exit_reason:
                    if direction == 'long':
                        do_sell(contract, qty, entry_price, price, rsi_now, exit_reason, trade_id)
                    else:
                        do_cover(contract, qty, entry_price, price, rsi_now, exit_reason, trade_id)

                    print(f"  🚨 EXIT TRIGGER: {exit_msg}")
                else:
                    if direction == 'long':
                        print(
                            f"  Holding long — no exit condition hit "
                            f"(hard stop {HARD_STOP_LOSS}, RSI target {LONG_EXIT_RSI})"
                        )
                    else:
                        print(
                            f"  Holding short — no exit condition hit "
                            f"(hard stop {HARD_STOP_LOSS}, RSI target {SHORT_EXIT_RSI})"
                        )

                await asyncio.sleep(0.5)
                continue

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            # No new trade if daily regime is unavailable / neutral
            if regime is None:
                print("  No entry — daily regime unavailable.")
                await asyncio.sleep(0.5)
                continue

            if regime == 'NEUTRAL':
                print("  No entry — RSI daily range is NEUTRAL.")
                await asyncio.sleep(0.5)
                continue

            today = datetime.now().date()
            if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                print(f"  Already traded {symbol} today — skipping.")
                await asyncio.sleep(0.5)
                continue

            in_long_zone = LONG_ENTRY_LOW <= rsi_now <= LONG_ENTRY_HIGH
            in_short_zone = SHORT_ENTRY_LOW <= rsi_now <= SHORT_ENTRY_HIGH

            qty = calculate_order_size(price)
            if qty <= 0:
                print(f"  Invalid order size for {symbol} at price ${price:.2f} — skipping.")
                await asyncio.sleep(0.5)
                continue

            position_value = qty * price

            # ── LONG: uptrend + RSI pulls back to 40–50 support ──────────
            if regime == 'UPTREND' and in_long_zone:
                ib.placeOrder(contract, day_market_order('BUY', qty))

                doc = create_trade_doc(symbol, 'long', price, qty, rsi_now, regime, sma200)
                trades_collection.insert_one(sanitize_for_mongo(doc))
                exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                print(f"  🚀 BUY (long)  {symbol}")
                print(f"     Entry:    ${price:.2f}  |  Regime=UPTREND")
                print(f"     Shares:   {qty}")
                print(f"     Notional: ${position_value:.2f}")
                print(f"     RSI:      {rsi_now:.1f}  (pullback to 40–50 support zone)")
                print(f"     Risk:     Hard stop at ${HARD_STOP_LOSS:.2f}")
                print(f"     Target:   RSI ≥ {LONG_EXIT_RSI}")

            # ── SHORT: downtrend + RSI rallies to 50–60 resistance ───────
            elif regime == 'DOWNTREND' and in_short_zone:
                ib.placeOrder(contract, day_market_order('SELL', qty))

                doc = create_trade_doc(symbol, 'short', price, qty, rsi_now, regime, sma200)
                trades_collection.insert_one(sanitize_for_mongo(doc))
                exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                print(f"  🔻 SELL (short)  {symbol}")
                print(f"     Entry:    ${price:.2f}  |  Regime=DOWNTREND")
                print(f"     Shares:   {qty}")
                print(f"     Notional: ${position_value:.2f}")
                print(f"     RSI:      {rsi_now:.1f}  (rally to 50–60 resistance zone)")
                print(f"     Risk:     Hard stop at ${HARD_STOP_LOSS:.2f}")
                print(f"     Target:   RSI ≤ {SHORT_EXIT_RSI}")

            else:
                if regime == 'UPTREND':
                    print(
                        f"  No entry — UPTREND but RSI={rsi_now:.1f} "
                        f"(waiting for 40–50 pullback zone)"
                    )
                else:
                    print(
                        f"  No entry — DOWNTREND but RSI={rsi_now:.1f} "
                        f"(waiting for 50–60 rally zone)"
                    )

            await asyncio.sleep(0.5)

        print(f"\n{'='*60}")
        print("  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  WMT  06:50:00
  SMA200=$109.56  DailyClose=$122.49  Regime=DOWNTREND
  Price=$124.06  RSI=50.4  RSI_prev=50.4
  OPEN SHORT  entry=$122.68  qty=32
    Current : $124.06  P&L=-1.12%  ($-44.16)
    Peak    : $122.21  MaxProfit=$+15.04
    Trough  : $124.49  MaxLoss  =$-57.92
  Holding short — no exit condition hit (hard stop -100.0, RSI target 30)

  MU  06:50:02
  SMA200=$245.66  DailyClose=$377.58  Regime=DOWNTREND
  Price=$414.01  RSI=47.1  RSI_prev=51.8
  No entry — DOWNTREND but RSI=47.1 (waiting for 50–60 rally zone)


Error 200, reqId 78: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 79: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  06:50:03
  Daily data unavailable / insufficient.
  Insufficient intraday data — skipping SAVA

  SLDB  06:50:03
  SMA200=$5.88  DailyClose=$7.74  Regime=UPTREND
  Price=$7.73  RSI=60.9  RSI_prev=60.9
  No entry — UPTREND but RSI=60.9 (waiting for 40–50 pullback zone)

  SLV  06:50:04
  SMA200=$53.17  DailyClose=$65.94  Regime=DOWNTREND
  Price=$69.63  RSI=55.9  RSI_prev=58.6
  🔻 SELL (short)  SLV
     Entry:    $69.63  |  Regime=DOWNTREND
     Shares:   57
     Notional: $3968.91
     RSI:      55.9  (rally to 50–60 resistance zone)
     Risk:     Hard stop at $-100.00
     Target:   RSI ≤ 30

  NEM  06:50:05
  SMA200=$90.71  DailyClose=$114.65  Regime=UPTREND
  Price=$121.17  RSI=54.6  RSI_prev=54.6
  No entry — UPTREND but RSI=54.6 (waiting for 40–50 pullback zone)

  CTXR  06:50:06
  SMA200=$1.15  DailyClose=$0.79  Regime=DOWNTREND
  Price=$0.81  RSI=78.2  RSI_prev=78.2
  No entry — DOWNTREND but RSI=78.2 (waiting for 50–60 rally zone)

  ONON  06:50:07
  SMA200=$44.83  Da

Error 200, reqId 138: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Error 200, reqId 139: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


...

  Daily data unavailable / insufficient.


reqHistoricalData: Timeout for Stock(conId=530319709, symbol='GRAB', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='GRAB', tradingClass='NMS')
Error 366, reqId 13360: No historical data query found for ticker id:13360, contract: Stock(conId=530319709, symbol='GRAB', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='GRAB', tradingClass='NMS')


  Insufficient intraday data — skipping GRAB

  RBLX  19:16:58


Peer closed connection.
Traceback (most recent call last):
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\ib_insync\util.py", line 341, in run
    result = loop.run_until_complete(task)
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\nest_asyncio.py", line 98, in run_until_complete
    return f.result()
           ~~~~~~~~^^
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\futures.py", line 194, in result
    raise self._make_cancelled_error()
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\tasks.py", line 306, in __step_run_and_handle_result
    result = coro.throw(exc)
  File "C:\Users\vrajp\anaconda3\Lib\site-packages\ib_insync\ib.py", line 1986, in reqHistoricalDataAsync
    await task
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\tasks.py", line 507, in wait_for
    return await fut
           ^^^^^^^^^
  File "C:\Users\vrajp\anaconda3\Lib\asyncio\futures.py", line 286, in __await__
    yield self  # This tells Task to wait for completion.
    ^^^^^^^^^^
asyncio.exceptions.Cancelle

Error: Socket disconnect
Disconnected from IB.


2026-04-08 19:36:19,113 [INFO] Connecting to 127.0.0.1:7497 with clientId 9575...
2026-04-08 19:36:19,115 [INFO] Connected
2026-04-08 19:36:19,119 [INFO] Logged on to server version 176
2026-04-08 19:36:19,121 [INFO] Warning 2104, reqId -1: Market data farm connection is OK:usfarm.nj
2026-04-08 19:36:19,122 [INFO] Warning 2104, reqId -1: Market data farm connection is OK:usopt.nj
2026-04-08 19:36:19,123 [INFO] API connection ready
2026-04-08 19:36:19,126 [INFO] Warning 2104, reqId -1: Market data farm connection is OK:usfarm
2026-04-08 19:36:19,127 [INFO] Warning 2106, reqId -1: HMDS data farm connection is OK:euhmds
2026-04-08 19:36:19,129 [INFO] Warning 2106, reqId -1: HMDS data farm connection is OK:fundfarm
2026-04-08 19:36:19,130 [INFO] Warning 2106, reqId -1: HMDS data farm connection is OK:ushmds
2026-04-08 19:36:19,132 [INFO] Warning 2158, reqId -1: Sec-def data farm connection is OK:secdefnj
2026-04-08 19:36:19,135 [INFO] position: Position(account='DUD087866', contract=Stock(

ValueError: Invalid format specifier '.4f if price else 'N/A'' for object of type 'float'

2026-04-08 19:38:35,635 [INFO] updatePortfolio: PortfolioItem(contract=Stock(conId=4391, symbol='AMD', right='0', primaryExchange='NASDAQ', currency='USD', localSymbol='AMD', tradingClass='NMS'), position=37.0, marketPrice=229.80000305, marketValue=8502.6, averageCost=232.095, unrealizedPNL=-84.91, realizedPNL=354.8, account='DUD087866')
2026-04-08 19:38:35,636 [INFO] updatePortfolio: PortfolioItem(contract=Stock(conId=5516, symbol='CCL', right='0', primaryExchange='NYSE', currency='USD', localSymbol='CCL', tradingClass='CCL'), position=0.0, marketPrice=27.8600006, marketValue=0.0, averageCost=0.0, unrealizedPNL=0.0, realizedPNL=-571.45, account='DUD087866')
2026-04-08 19:38:35,638 [INFO] updatePortfolio: PortfolioItem(contract=Stock(conId=9939, symbol='MU', right='0', primaryExchange='NASDAQ', currency='USD', localSymbol='MU', tradingClass='NMS'), position=100.0, marketPrice=402.2999878, marketValue=40230.0, averageCost=410.05585, unrealizedPNL=-775.59, realizedPNL=364.48, account='DU